In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain.output_parsers import PydanticOutputParser
from langchain_google_genai import GoogleGenerativeAI
from langchain.prompts import PromptTemplate
from dotenv import load_dotenv
from os import getenv

load_dotenv()

from os.path import dirname, abspath
from sys import path

SCRIPT_DIR = dirname(abspath(__name__))
path.append(dirname(SCRIPT_DIR))

from models.linkedin import Job

import pandas as pd

In [ ]:
prompt = PromptTemplate(template="Transform the following data by translating into English, summarizing the position and location names and converting the date, using the column 'last_status', for a yyyy-mm-dd format: {JOB}\n\n{FMT}")
LLM = GoogleGenerativeAI(model="gemini-2.0-flash", api_key=getenv("GOOGLE_API_KEY"), temperature=0)
chain = (prompt | LLM | (parser := PydanticOutputParser(pydantic_object=Job)))
data = CSVLoader("data/raw.csv").load()

In [ ]:
response = [chain.invoke({"JOB": d.page_content, "FMT": parser.get_format_instructions()}) for d in data[:18]]

In [ ]:
df = pd.json_normalize([r.model_dump() for r in response])
df.to_csv("data/silver.csv", sep=";", index=False)